# DASYNET: PneumoniaMNIST Classification

This notebook implements **DASYNET**—a custom Convolutional Neural Network (CNN) built from scratch to classify chest X-rays (normal vs. pneumonia) from the MedMNIST collection.

The goal is to match benchmark-level accuracy (typically ~90-95% for simple lightweight CNNs on this dataset) using a logical sequence of Convolution, Batch Normalization, and Max Pooling layers.

In [1]:
# Colab setup: install MedMNIST + CUDA-enabled PyTorch builds
# If this cell upgrades torch, restart runtime and run all cells from top.
!pip -q install -U medmnist
!pip -q install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

import torch

print("torch:", torch.__version__)
print("torch cuda build:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))
else:
    print("No CUDA GPU detected by runtime. In Colab, set Runtime > Change runtime type > T4 GPU.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.1 MB/s eta 0:00:00
torch: 2.9.0+cpu
torch cuda build: None
cuda available: False
No CUDA GPU detected by runtime. In Colab, set Runtime > Change runtime type > T4 GPU.


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
import medmnist
from medmnist import INFO, Evaluator
from torch.utils.data import DataLoader

print(f"MedMNIST v{medmnist.__version__} @ {medmnist.__path__}")


def resolve_device() -> torch.device:
    """Use CUDA only if a tiny CUDA op succeeds; otherwise fallback to CPU."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        _ = torch.randn(1, device="cuda")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA unavailable at runtime ({e}). Falling back to CPU.")
        return torch.device("cpu")


# Check device compatibility
device = resolve_device()
print(f"Using device: {device}")

# 2. Data Loading and Preprocessing
data_flag = 'pneumoniamnist'
download = True
IMG_SIZE = 224

# Use a smaller batch size on CPU to avoid memory/latency issues.
BATCH_SIZE = 128 if device.type == 'cuda' else 32

info = INFO[data_flag]
n_classes = len(info['label'])
DataClass = getattr(medmnist, info['python_class'])

# Define transformations for 224x224 grayscale inputs
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load datasets at 224x224
train_dataset = DataClass(split='train', transform=data_transform, download=download, size=IMG_SIZE)
val_dataset = DataClass(split='val', transform=data_transform, download=download, size=IMG_SIZE)
test_dataset = DataClass(split='test', transform=data_transform, download=download, size=IMG_SIZE)

loader_kwargs = {
    "num_workers": 2,
    "pin_memory": device.type == 'cuda'
}

# Create Data Loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
# Use non-shuffled loaders for metric computation with MedMNIST evaluator
train_eval_loader = DataLoader(dataset=train_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)
val_loader = DataLoader(dataset=val_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(dataset=test_dataset, batch_size=2 * BATCH_SIZE, shuffle=False, **loader_kwargs)

print(f"Training set: {len(train_dataset)} images")
print(f"Test set: {len(test_dataset)} images")
print(f"Batch size: {BATCH_SIZE}")

# 3. DASYNET Architecture
class DASYNET(nn.Module):
    def __init__(self, in_channels=1, num_classes=n_classes):
        super(DASYNET, self).__init__()

        # Block 1 - "Scanner": 224x224 -> 112x112
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 2 - "Scanner": 112x112 -> 56x56
        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Block 3 - "Scanner": 56x56 -> 28x28
        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Keep classifier dimensions stable across input sizes.
        self.global_pool = nn.AdaptiveAvgPool2d((3, 3))
        self.flatten = nn.Flatten()

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(64 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

model = DASYNET(in_channels=1, num_classes=n_classes).to(device)

# 4. Initialization, Optimizer, and Loss
EPOCHS = 35
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# 5. Model Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device).squeeze().long()

        outputs = model(inputs)
        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += outputs.max(1)[1].eq(targets).sum().item()
        total += targets.size(0)

    epoch_train_loss = running_loss / total
    epoch_train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device).squeeze().long()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            val_loss += loss.item() * inputs.size(0)
            val_correct += outputs.max(1)[1].eq(targets).sum().item()
            val_total += targets.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = 100. * val_correct / val_total
    scheduler.step(epoch_val_loss)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {epoch_train_loss:.4f}, Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f}, Acc: {epoch_val_acc:.2f}%")

# 6. Evaluation
def test(split):
    model.eval()
    y_score = torch.tensor([])

    if split == 'train':
        loader = train_eval_loader
    elif split == 'test':
        loader = test_loader
    else:
        raise ValueError("split must be 'train' or 'test'")

    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            scores = F.softmax(outputs, dim=1)
            y_score = torch.cat((y_score, scores.cpu()), 0)

    metrics = medmnist.Evaluator(data_flag, split, size=IMG_SIZE).evaluate(y_score.numpy())
    print(f'{split.capitalize()} Set AUC: {metrics[0]:.4f} | Accuracy: {metrics[1]:.4f}')

print("\n--- Final Benchmarking ---")
test('train')
test('test')

MedMNIST v3.0.2 @ ['/usr/local/lib/python3.12/dist-packages/medmnist']
Using device: cpu


100%|██████████| 214M/214M [00:17<00:00, 12.2MB/s] 


Training set: 4708 images
Test set: 624 images
Batch size: 32
Epoch [1/35] Train Loss: 0.2559, Acc: 89.23% | Val Loss: 0.3882, Acc: 80.53%
Epoch [5/35] Train Loss: 0.1186, Acc: 95.71% | Val Loss: 0.1108, Acc: 96.37%
Epoch [10/35] Train Loss: 0.0930, Acc: 96.37% | Val Loss: 0.0834, Acc: 95.80%
Epoch [15/35] Train Loss: 0.0874, Acc: 96.47% | Val Loss: 0.0769, Acc: 97.14%
Epoch [20/35] Train Loss: 0.0645, Acc: 97.47% | Val Loss: 0.1345, Acc: 93.89%
Epoch [25/35] Train Loss: 0.0573, Acc: 97.68% | Val Loss: 0.0834, Acc: 96.56%
Epoch [30/35] Train Loss: 0.0374, Acc: 98.51% | Val Loss: 0.0556, Acc: 98.28%
Epoch [35/35] Train Loss: 0.0386, Acc: 98.68% | Val Loss: 0.1554, Acc: 95.04%

--- Final Benchmarking ---
Train Set AUC: 0.9990 | Accuracy: 0.9601
Test Set AUC: 0.9541 | Accuracy: 0.7500


In [ ]:
# Save trained weights with robust Colab fallbacks
import os
import shutil
import torch

model_path = 'dasynet_pneumonia.pth'
torch.save(model.state_dict(), model_path)

abs_path = os.path.abspath(model_path)
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Saved model to: {abs_path}")
print(f"File size: {size_mb:.2f} MB")

# Try browser download (works in standard Colab browser sessions).
try:
    from google.colab import files
    files.download(model_path)
    print('Download trigger sent. If nothing appears, use the Drive copy path below.')
except Exception as e:
    print(f'Browser download not available in this client: {e}')

# Always create a persistent Drive copy when available.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_dir = '/content/drive/MyDrive/IDS705'
    os.makedirs(drive_dir, exist_ok=True)
    drive_path = os.path.join(drive_dir, model_path)
    shutil.copy2(model_path, drive_path)
    print(f'Drive backup saved to: {drive_path}')
except Exception as e:
    print(f'Drive backup skipped: {e}')

print('Done. If browser download did not start, get the file from MyDrive/IDS705.')

Model successfully saved to /content/dasynet_pneumonia.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started in browser.
